In [ ]:

#Importing Libraries
import os
import cv2
import numpy as np
import random
import gc
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    DepthwiseConv2D, Conv2D, BatchNormalization,
    MaxPooling2D, Flatten, Dense, Dropout,
    Input, TimeDistributed, GRU, GlobalAveragePooling2D
)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)


In [ ]:
#preparing dataset to the required format
IMG_SIZE = (128, 128)
MAX_FRAMES = 100        
SKIP = 3              
SEQ_LEN = 10
dataset_path = "/kaggle/input/datasets/abc123456/crowddataset/dataset"
frame_path = "/kaggle/working/frames"
import os
base_path = "/kaggle/input/datasets/abc123456/crowddataset/dataset/"
for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".mp4"):
            print(os.path.join(root, file))

In [ ]:
#frame extraction
def extract_frames(dataset_path, output_path):
    for split in ["train", "test"]:
        for label in ["normal", "abnormal"]:
            input_folder = os.path.join(dataset_path, split, label)
            for video_file in os.listdir(input_folder):
                if not video_file.endswith(".mp4"):
                    continue
                cap = cv2.VideoCapture(os.path.join(input_folder, video_file))
                save_folder = os.path.join(output_path, split, label, video_file.split(".")[0])
                os.makedirs(save_folder, exist_ok=True)
                count = 0
                saved = 0
                while True:
                    ret, frame = cap.read()
                    if not ret:
                        break
                    if count % SKIP == 0:
                        frame = cv2.resize(frame, IMG_SIZE)
                        cv2.imwrite(
                            os.path.join(save_folder, f"frame_{saved:04d}.jpg"),
                            frame
                        )
                        saved += 1
                        if saved >= MAX_FRAMES:
                            break
                    count += 1
                cap.release()
                print(f"{video_file} → {saved} frames")
extract_frames(dataset_path, frame_path)
print("Frames extracted")


In [ ]:
#Loading Frames
def load_frames(base_path):
    X, y = [], []
    for label, val in zip(["normal", "abnormal"], [0,1]):
        label_path = os.path.join(base_path, "train", label)
        for video in os.listdir(label_path):
            video_path = os.path.join(label_path, video)
            frames = sorted(os.listdir(video_path))[:MAX_FRAMES]
            for f in frames:
                img = cv2.imread(os.path.join(video_path, f))
                img = cv2.resize(img, IMG_SIZE)
                img = img / 255.0
                X.append(img)
                y.append(val)
    return np.array(X), np.array(y)
X, y = load_frames(frame_path)
X, y = shuffle(X, y)
print(X.shape, y.shape)


In [ ]:
#Balancing
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)
class_weights = dict(enumerate(class_weights))
print(class_weights)


In [ ]:
#CNN training
cnn_model = Sequential([
    Input(shape=(128,128,3)),

    DepthwiseConv2D((3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(),

    DepthwiseConv2D((3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(32, (1,1), activation='relu'),

    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])
cnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)
cnn_model.fit(
    X, y,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=[early_stop]
)


In [ ]:
#CNN model saving
A_t = cnn_model.predict(X).flatten()
np.save("/kaggle/working/A_t.npy", A_t)
print("A_t saved")
cnn_model.save("/kaggle/working/cnn_model.h5")
print("CNN model saved")



In [ ]:
#Load frames
def load_video_frames(base_path):
    videos, labels = [], []
    for label, val in zip(["normal", "abnormal"], [0,1]):
        label_path = os.path.join(base_path, "train", label)
        for video in os.listdir(label_path):
            video_path = os.path.join(label_path, video)
            frames = sorted(os.listdir(video_path))[:MAX_FRAMES]
            vid_frames = []
            for f in frames:
                img = cv2.imread(os.path.join(video_path, f))
                img = cv2.resize(img, IMG_SIZE)
                img = img / 255.0
                vid_frames.append(img)
            videos.append(np.array(vid_frames))
            labels.append(val)
    return videos, labels
videos, labels = load_video_frames(frame_path)


In [ ]:
#Optical Flow
def compute_flow(frames):
    flows = []
    prev = cv2.cvtColor((frames[0]*255).astype(np.uint8), cv2.COLOR_BGR2GRAY)
    for i in range(1, len(frames)):
        curr = cv2.cvtColor((frames[i]*255).astype(np.uint8), cv2.COLOR_BGR2GRAY)
        flow = cv2.calcOpticalFlowFarneback(
            prev, curr, None,
            0.5, 3, 15, 3, 5, 1.2, 0
        )
        mag, ang = cv2.cartToPolar(flow[...,0], flow[...,1])
        mag = mag / (np.max(mag)+1e-6)
        ang = ang / (np.max(ang)+1e-6)
        flows.append(np.stack([mag, ang], axis=-1))
        prev = curr
    return np.array(flows)


In [ ]:
#sequence creation
def create_sequences(flow):
    X = []
    for i in range(len(flow)-SEQ_LEN):
        X.append(flow[i:i+SEQ_LEN])
    return np.array(X)
all_flows = []
all_labels = []
for vid, label in zip(videos, labels):
    if len(vid) < 2:
        continue
    flow = compute_flow(vid)
    all_flows.append(flow)
    all_labels.append(label)
X_seq, y_seq = [], []
flow_memory = []
frame_index_memory = []
global_idx = 0 
for flow, label in zip(all_flows, all_labels):
    X_temp = create_sequences(flow)
    for seq in X_temp:
        X_seq.append(seq)
        y_seq.append(label)
        # explanation storage
        flow_memory.append(seq[-1])
        frame_index_memory.append(global_idx)
        global_idx += 1
X_seq = np.array(X_seq)
y_seq = np.array(y_seq)
flow_memory = np.array(flow_memory)
frame_index_memory = np.array(frame_index_memory)
print(X_seq.shape, y_seq.shape)
X_train, X_val, y_train, y_val = train_test_split(
    X_seq, y_seq,
    test_size=0.2,
    random_state=42,
    stratify=y_seq
)



In [ ]:
#GRU training
model = Sequential()
model.add(Input(shape=(SEQ_LEN,128,128,2)))
model.add(TimeDistributed(Conv2D(32,3,activation='relu')))
model.add(TimeDistributed(MaxPooling2D()))
model.add(TimeDistributed(Conv2D(64,3,activation='relu')))
model.add(TimeDistributed(MaxPooling2D()))
model.add(TimeDistributed(GlobalAveragePooling2D()))
model.add(GRU(64))
model.add(Dropout(0.5))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=3,
    batch_size=2,
    callbacks=[early_stop]
)


In [ ]:
#GRU model saving
M_t = model.predict(X_seq).flatten()
np.save("/kaggle/working/M_t.npy", M_t)
print("M_t saved")
model.save("/kaggle/working/model.h5")
print("GRU model saved")